<p align="center">
  <img src="https://img.shields.io/badge/Research%20Mode-ON-4cbb17?style=for-the-badge" alt="Research Mode">
</p>


# Substantia Nigra Cell Subtyping 
## Case Study — Part 02: Preprocessing and Feature Selection

**ASAP CRN Learning Lab**  
Reproducible exploratory and meta-analysis examples using ASAP CRN data

---

### Overview

This notebook continues the Substantia Nigra case study by applying standardized preprocessing and feature selection to the AnnData artifacts generated in Part 01. These steps prepare the data for downstream integration, clustering, and cell-type annotation.

---

### Learning Objectives

By the end of this notebook, you will be able to:

- Normalize and log-transform single-cell expression data
- Identify and subset highly variable genes (HVGs)
- Generate an analysis-ready AnnData object for downstream workflows
---


### Prerequisites

- Completion of Case Study — Part 01: Set Up and Data Preparation
- Access to the previously generated Substantia Nigra AnnData artifacts
- A configured analysis environment with required dependencies available
---

### Inputs

- Curated Substantia Nigra AnnData object (full gene space)
    - Example: `asap-{dataset_team}__sn_cells__full_genes__curated.h5ad`
---

### Outputs

This notebook generates the following AnnData artifacts:

- **Analysis-ready Substantia Nigra AnnData object**
    - Example: asap-{dataset_team}__sn_cells__preprocessed.h5ad
    - Contains:
        - Substantia Nigra–derived cells
        - Normalized and log-transformed expression values
        - Preprocessing outputs (e.g., PCA stored in .obsm)
        - ENSG IDs in .var_names for compatibility with MapMyCells

> This artifact is designed to be reused directly in downstream notebooks focused on integration, clustering, and cell-type annotation.

---

### Notes

- Parameters may be adapted for exploratory analyses; however, downstream notebooks assume the default outputs generated here.


## Table of Contents

1. [Package Imports and Configuration](#2-package-imports-and-configuration)
2. [Data Sources and Context](#3-data-sources-and-context)
4. [Data Preproccesing](#4-data-preprocessing)
5. [Data Export](#5-data-export)

## 1. Package Imports and Configuration

In [ ]:
# Core scientific computing and visualization libraries
import numpy as np
import pandas as pd
import scanpy as sc

# Standard library imports
import sys
import subprocess
import importlib
import warnings
import os
from pathlib import Path

# Optional: enable cell-level timing for performance awareness
try:
    %load_ext autotime
except ModuleNotFoundError:
    %pip install ipython-autotime
    %load_ext autotime


## 2. Data sources and Context

### 2.1 Set dataset paths
In this example, we are working with the **PMDBS single‑cell RNA‑seq cohort** dataset:

- **Workflow** → `pmdbs_sc_rnaseq`  
- **Team** → `cohort`  
- **Source** → `pmdbs`  
- **Type** → `sc-rnaseq`  

These components are combined to construct the bucket and dataset names.  
We then set the path to the **cohort analysis outputs** and preview the available files.


In [ ]:
#set general folder paths
HOME = Path.home()
WS_ROOT = HOME / "workspace"
DATA_DIR = WS_ROOT / "Data"
WS_FILES = WS_ROOT / "ws_files"

if not WS_ROOT.exists():
    print(f"{WS_ROOT} doesn't exist. We need to remount our resources")
    !wb resource mount    

print("Home directory:     ", HOME)
print("Workspace root:     ", WS_ROOT)
print("Data directory:     ", DATA_DIR)
print("ws_files directory: ", WS_FILES)

print("\nContents of workspace root:")
for p in WS_ROOT.glob("*"):
    print(" -", p.name, "/" if p.is_dir() else "")

In [ ]:
from pathlib import Path

In [ ]:
## Build and set path to desired dataset

DATASETS_PATH = WS_ROOT / "01_PMDBS"

workflow       = "pmdbs_sc_rnaseq" 
dataset_team   = "cohort"
dataset_source = "pmdbs"
dataset_type   = "sc-rnaseq"

bucket_name  = f"asap-curated-{dataset_team}-{dataset_source}-{dataset_type}"

dataset_path = DATASETS_PATH / "PMDBS_sc_rnaseq" / bucket_name / workflow
print("Dataset Path:", dataset_path)

cohort_analysis_path = dataset_path / "cohort_analysis"

# Define the local path for case study output
local_data_path = WS_FILES / "sn_celltyping"
!ls {local_data_path}

## 3. Data Preprocessing

In this section, we load the Substantia Nigra–restricted AnnData object generated in Part 01 and prepare it for feature selection. We explicitly preserve raw counts, apply normalization and log-transformation, and compute highly variable genes (HVGs) using a sample-aware strategy.

### 3.1 Load Substantia Nigra AnnData Object

In [ ]:
# Load curated Substantia Nigra AnnData object (full gene space)
sn_full_raw_filename = (
    local_data_path / f"asap-{dataset_team}.sn_cells__full_genes_curated.h5ad"
)
adata = sc.read_h5ad(sn_full_raw_filename)

> ⏱️ **Expected runtime:** ~6 minutes depending on dataset size and available compute when run for the first time.

### 3.2 Preserve Raw Counts and Expression State
We explicitly store raw counts and the unprocessed expression matrix to support reproducibility and downstream reuse.

In [ ]:
# Preserve raw counts and original expression matrix
adata.layers['counts'] = adata.X.copy()

### 3.3 Normalize and Log-Transform Expression
Expression values are normalized to a fixed library size and log-transformed. These transformed values are used for HVG detection and downstream dimensionality reduction, while raw counts remain accessible.

In [ ]:
# Normalize and log-transform expression values
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
# Save the normalized-log data
adata.layers['lognorm']=adata.X.copy() 

### 3.4 Recompute Feature Selection and PCA
Because this dataset was generated by subsetting Substantia Nigra cells from a larger whole–brain dataset while retaining the full gene space, the variance structure of the data changes. Highly variable genes and principal components computed on the full dataset reflect global brain-wide variation and are therefore not appropriate for analyses restricted to the Substantia Nigra subset. For this reason, both HVG selection and PCA are recomputed to capture biologically meaningful variation specific to Substantia Nigra cells.

In [ ]:
# Detect highly variable genes (HVGs) within the Substantia Nigra subset
n_top_genes = 3000
sc.experimental.pp.highly_variable_genes(
        adata,
        n_top_genes=n_top_genes,
        batch_key="sample", #using sample not batch
        flavor="pearson_residuals",
        check_values=True,
        layer="counts",
        subset=False,
        inplace=True
)

In [ ]:
# Double check that no transcripts not found in cells are in the atlas
min_cells = 5
sc.pp.filter_genes(adata, min_cells=min_cells)

In [ ]:
# Build a kNN graph in the scVI latent space to capture denoised,
# batch-corrected cellular relationships prior to Leiden clustering.
sc.pp.neighbors(adata, use_rep="_X_scVI", n_neighbors=15, transformer="pynndescent", method = "umap")
sc.tl.leiden(adata, resolution=1.0, key_added="leiden_sn_scVI_1.0", random_state=0)

> ⏱️ **Expected runtime:** ~30 minutes depending on dataset size and available compute when run for the first time.

## 4. Prepare Data for Export
In this step, we clean and export the analysis-ready Substantia Nigra AnnData object generated in this notebook. This artifact reflects all preprocessing, feature selection, and dimensionality reduction steps applied after subsetting to Substantia Nigra cells.

The exported AnnData object serves as a reproducible input for subsequent parts of the case study and supports downstream integration, clustering, and cell-type annotation workflows.

This AnnData object contains:

- Cells restricted to the Substantia Nigra (SN)
- Expression values that have been normalized and log-transformed in .layers["lognorm"]
- Highly variable genes (HVGs) used for downstream modeling
- Raw counts preserved in .layers["counts"]
- Preprocessing outputs (e.g., PCA, latent embeddings) stored in .obsm
- Curated metadata and annotations stored in .obs
- ENSG IDs in .var_names for compatibility with MapMyCells

### 4.1 Remove bulky objects from Anndata 

In order to improve efficiency we will remove some heavy object from our adata object that we will not need.

In [ ]:
# Drop large neighbor matrices
for k in ['neighbors_scvi_distances','neighbors_scvi_connectivities']:
    adata.obsp.pop(k, None)


# Drop heavy per-gene stats (keep only highly_variable + n_cells)
adata.var.drop(columns=[
    'means','variances','residual_variances',
    'highly_variable_rank','highly_variable_nbatches','highly_variable_intersection'
], errors='ignore', inplace=True)

### 4.2 Make Compatible MapMyCells 

Before running MapMyCells, the query dataset must be formatted to match the reference mapping requirements.

### Why this step is needed
MapMyCells expects gene features as **ENSG-style Ensembl IDs** in the row names, rather than gene symbols. Many single-cell datasets use gene symbols by default, so we standardize them here while the dataset is in memory to streamline downstream steps.

Without consistent ENSG identifiers, mapping may fail or result in incomplete gene matching.

In [ ]:
# Preserve gene_name
adata.var["gene_name"] = adata.var_names

# Set var_names to ENSG IDs
adata.var_names = adata.var["gene_id"].astype(str)

# Ensure uniqueness
adata.var_names_make_unique()

# Quick check
print(adata.var.head())
print(adata.var_names[:5])


In [ ]:
sn_processed_filename = (
    local_data_path / f"asap-{dataset_team}_sn_cells__preprocessed.h5ad"
)


adata.write_h5ad(sn_processed_filename)

> ⏱️ **Expected runtime:** ~20 minutes depending on dataset size and available compute when run for the first time.

---

## Summary and Next Steps

In this notebook, we prepared Substantia Nigra–restricted single-cell data for downstream analysis by applying standardized preprocessing, feature selection, and dimensionality reduction. These steps ensure that subsequent analyses capture biologically meaningful variation specific to Substantia Nigra cells.

The exported AnnData artifact provides a stable, reproducible handoff for downstream workflows, including integration, clustering, and cell-type annotation.

### Continue the Case Study

- Proceed to **Part 03: MapMyCells Mapping**
- Refer to the **ASAP-CRN Learning Lab documentation** for additional context, workflows, and best practices.

> This notebook is part of the ASAP-CRN Learning Lab and is intended to be executed using approved data accessed through the ASAP-CRN Cloud.

